## 1. Setup

### 1.1 Load, scope and split

The same steps as `01_scoping_eda.ipynb` (sections 2 and 3.1), without the checks and explanations. The seed and the
row order are the same, so train, validation and test contain the same patients the EDA decisions were based on.


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data/raw")  # notebook runs from notebooks/
DIAG_COLS = ["diag_1", "diag_2", "diag_3"]

# Discharged as expired (11, 19, 20, 21) or to hospice (13, 14)
EXCLUDED_DISCHARGE = [11, 13, 14, 19, 20, 21]


def icd9_group(code: str | float) -> str:
    """Map an ICD-9 code to one of the 9 disease groups of Strack et al. (2014)."""
    if pd.isna(code):
        return "Missing"
    if code[0] in "VE":  # supplementary (V) and external-cause (E) codes
        return "Other"
    value = float(code)
    chapter = int(value)  # integer part: "786.05" -> 786
    if 390 <= value < 460 or chapter == 785:
        return "Circulatory"
    if 460 <= value < 520 or chapter == 786:
        return "Respiratory"
    if 520 <= value < 580 or chapter == 787:
        return "Digestive"
    if chapter == 250:
        return "Diabetes"
    if 800 <= value < 1000:
        return "Injury"
    if 710 <= value < 740:
        return "Musculoskeletal"
    if 580 <= value < 630 or chapter == 788:
        return "Genitourinary"
    if 140 <= value < 240:
        return "Neoplasms"
    return "Other"


df_raw = pd.read_csv(
    DATA_DIR / "diabetic_data.csv",
    na_values=["?"],        # the dataset's only missing-value token
    keep_default_na=False,  # keep "None" (test not measured) as a category
    low_memory=False,
)

# Scoping: first encounter per patient, no death / hospice, known gender
df = df_raw.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")
df = df[~df["discharge_disposition_id"].isin(EXCLUDED_DISCHARGE)]
df = df[df["gender"] != "Unknown/Invalid"].reset_index(drop=True)

# 1 = readmitted within 30 days; ">30" and "NO" both become 0
df["target"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns="readmitted")

# Empty code beyond number_diagnoses = no such diagnosis -> "Absent" instead of "Missing"
for position, col in enumerate(DIAG_COLS, start=1):
    groups = df[col].map(icd9_group)
    absent = df[col].isna() & (df["number_diagnoses"] < position)
    df[f"{col}_group"] = groups.mask(absent, "Absent")

# Same seed and order as the EDA, so the three parts contain the same patients
df_full_train, df_test = train_test_split(df, test_size=0.2, stratify=df["target"], random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, stratify=df_full_train["target"], random_state=1
)

for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"{name}: {len(part)} rows, positive rate {part['target'].mean():.2%}")


train: 41982 rows, positive rate 8.97%
val: 13994 rows, positive rate 8.98%
test: 13994 rows, positive rate 8.97%


## 2. Data preparation

### 2.1 Fixed cleaning rules

The decisions from section 3.6 of the EDA. A function, so that train, validation and later the service all get exactly the same
treatment. These rules are fixed in advance and read nothing from the data, so they cannot leak information from validation.


In [3]:
DROP_COLS = [
    "encounter_id", "patient_nbr",  # identifiers
    "examide", "citoglipton", "glimepiride-pioglitazone", "metformin-pioglitazone",  # one value in train
    "weight", "payer_code",  # mostly missing
    *DIAG_COLS,  # replaced by the _group columns
]

# Codes that mean "not available / NULL / not mapped" in IDS_mapping.csv
UNKNOWN_CODES = {
    "admission_type_id": [5, 6, 8],
    "discharge_disposition_id": [18, 25, 26],
    "admission_source_id": [9, 15, 17, 20, 21],
}


def clean(data: pd.DataFrame) -> pd.DataFrame:
    """Apply the fixed rules of the 3.6 decisions table. Nothing here is learned from the data."""
    data = data.drop(columns=DROP_COLS)
    data[["race", "medical_specialty"]] = data[["race", "medical_specialty"]].fillna("Missing")
    for col, codes in UNKNOWN_CODES.items():
        # Codes become text, so a model cannot treat them as quantities
        data[col] = data[col].astype(str).mask(data[col].isin(codes), "Unknown")
    return data


# New names, so df_train and df_val stay intact and this cell can be re-run
train_clean = clean(df_train)
val_clean = clean(df_val)

print(train_clean.shape, val_clean.shape)
print("missing values left:", train_clean.isna().sum().sum(), val_clean.isna().sum().sum())
train_clean[list(UNKNOWN_CODES)].eq("Unknown").sum()


(41982, 42) (13994, 42)
missing values left: 0 0


admission_type_id           4716
discharge_disposition_id    1960
admission_source_id         3035
dtype: int64

### 2.2 Merge rare categories

Categories with under 100 training patients become "Other", in `medical_specialty` and the three ID columns, because rates
measured on tiny groups are unreliable. Unlike 2.1, this step learns from the data: the lists of kept categories come from
the training set only, and the same lists are applied to validation and later saved for the service.


In [5]:
MIN_PATIENTS = 100
RARE_COLS = ["medical_specialty", *UNKNOWN_CODES]  # unpacking a dict gives its keys


def fit_kept_categories(train: pd.DataFrame) -> dict[str, list[str]]:
    """Return, per column, the categories with at least MIN_PATIENTS patients in the training data."""
    kept = {}
    for col in RARE_COLS:
        counts = train[col].value_counts()
        kept[col] = counts[counts >= MIN_PATIENTS].index.tolist()
    return kept


def merge_rare(data: pd.DataFrame, kept: dict[str, list[str]]) -> pd.DataFrame:
    """Replace every category outside the kept lists with "Other"."""
    data = data.copy()  # leave the caller's DataFrame untouched
    for col, categories in kept.items():
        # .where keeps the value when the condition is true, otherwise writes "Other"
        data[col] = data[col].where(data[col].isin(categories), "Other")
    return data


kept_categories = fit_kept_categories(train_clean)
train_prep = merge_rare(train_clean, kept_categories)
val_prep = merge_rare(val_clean, kept_categories)

pd.DataFrame({
    "before": train_clean[RARE_COLS].nunique(),
    "after": train_prep[RARE_COLS].nunique(),
    "train_to_other": train_prep[RARE_COLS].eq("Other").sum(),
    "val_to_other": val_prep[RARE_COLS].eq("Other").sum(),
})


,before,after,train_to_other,val_to_other
medical_specialty,66,22,916,320
admission_type_id,6,5,14,4
discharge_disposition_id,20,11,151,52
admission_source_id,15,8,100,35


### 2.3 Feature sets and target

Two lists of inputs go to modelling: all columns, and a reduced list without the 11 near-constant drugs and `gender`. The
reduced list goes into the service unless it lowers the validation AUC by more than 0.005 compared with all columns.


In [6]:
# 99%+ of train patients on one value, plus gender (MI 0): removed in the reduced list
LOW_SIGNAL_COLS = [
    "acetohexamide", "metformin-rosiglitazone", "troglitazone", "glipizide-metformin",
    "miglitol", "tolbutamide", "tolazamide", "chlorpropamide", "acarbose", "nateglinide",
    "glyburide-metformin", "gender",
]

FEATURES_ALL = [c for c in train_prep.columns if c != "target"]
FEATURES_REDUCED = [c for c in FEATURES_ALL if c not in LOW_SIGNAL_COLS]

# The 6 skewed counts: log1p is tested against plain values on the logistic regression only
LOG_COLS = [
    "time_in_hospital", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient",
]

y_train = train_prep["target"].to_numpy()
y_val = val_prep["target"].to_numpy()

# Catches a typo in a column name, which the list comprehension above would silently ignore
assert set(LOW_SIGNAL_COLS) <= set(FEATURES_ALL)
assert set(LOG_COLS) <= set(FEATURES_ALL)

print(f"all: {len(FEATURES_ALL)} inputs, reduced: {len(FEATURES_REDUCED)} inputs")
print(f"y_train mean {y_train.mean():.4f}, y_val mean {y_val.mean():.4f}")


all: 41 inputs, reduced: 29 inputs
y_train mean 0.0897, y_val mean 0.0898
